# 02 · Curve representation and identification

**RQ1:** Which risks survive compression into a smooth finite curve?
**Hypothesis:** good price-level reconstruction does not imply accurate spread
or butterfly risk, and increasing dimension may trade approximation error for
conditioning. This is separate from optimizer convergence.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from spde_covariance.empirical_research import load_research_evidence

evidence = load_research_evidence(Path("../experiments/results/shfe-research"))
print(evidence["protocol"]["status"])
print("Numerical source:", evidence["manifest"]["git_revision"])
print("Original baseline source:", evidence["protocol"]["baseline_source_revision"])
assert evidence["protocol"]["baseline_summary_sha256"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def table(rows, columns):
    print(" | ".join(columns))
    for row in rows:
        print(
            " | ".join(
                (
                    f"{row.get(c):.6g}" if isinstance(row.get(c), float) else str(row.get(c))
                    for c in columns
                )
            )
        )

## Exact transport and the residual identity

For the invariant Laguerre span, $E_{i+1}M_{\Delta_i}=E_i$ at the same contract
maturities. If $a_i$ denotes the fitted state, then

```math
\Delta p_i=E_{i+1}(a_{i+1}-M_{\Delta_i}a_i)+u_i,\qquad
Y_i=S_i+U_i.
```

Therefore raw sample second moments obey

```math
\overline{Y^2}=\overline{S^2}+\overline{U^2}+2\overline{SU}.
```

The signed cross term is retained. Residuals are realized endpoint
reconstruction-error changes, not fresh independent price noise. This is a
post hoc decomposition; future reconstructed states never enter a forecast.

In [ ]:
fits = evidence["representation"]["fits"]
table(
    [
        {
            "dimension": r["dimension"],
            "smoothing": r["smoothing"],
            "median_rmse": r["rmse_quantiles"][2],
            "median_condition": r["condition_quantiles"][1],
            "available": r["available_sessions"],
        }
        for r in fits
    ],
    ["dimension", "smoothing", "median_rmse", "median_condition", "available"],
)
for r in fits:
    for target in r["decomposition"]["exposures"]:
        if target["count"]:
            assert target["identity_relative_error"] < 1e-10

In [ ]:
base = next((r for r in fits if r["dimension"] == 3 and r["smoothing"] == 1e-06))
rows = base["decomposition"]["exposures"]
table(rows, ["exposure", "count", "modeled_share", "residual_share", "cross_share"])
fig, ax = plt.subplots(figsize=(8, 3))
x = np.arange(len(rows))
width = 0.24
for j, (key, label, color) in enumerate(
    [
        ("modeled_share", "Fitted curve", "0.25"),
        ("residual_share", "Residual", "0.6"),
        ("cross_share", "Signed cross", "0.85"),
    ]
):
    ax.bar(
        x + (j - 1) * width,
        [r[key] for r in rows],
        width,
        label=label,
        color=color,
        edgecolor="black",
        linewidth=0.4,
    )
ax.axhline(0, color="black", linewidth=0.6)
ax.set_xticks(x, [r["exposure"] for r in rows], rotation=12)
ax.set(
    ylabel="Share of realized second moment",
    title="Curve fit and financial risk are different targets",
)
ax.legend()
fig.tight_layout()
plt.show()

## A clean identification control

Suppose observations reveal covariance blocks on nodes $(1,2)$ and $(2,3)$,
but never the pair $(1,3)$. All matrices below are PSD and give the same observed
blocks. The unobserved spread risk is $2-2c\in[0,4]$. Regularization or canonical
completion chooses a matrix; the observations do not identify the choice.

In [ ]:
for c in (-1.0, 0.0, 1.0):
    K = np.array([[1.0, 0.0, c], [0.0, 1.0, 0.0], [c, 0.0, 1.0]])
    a = np.array([1.0, 0.0, -1.0])
    assert np.linalg.eigvalsh(K)[0] >= -1e-12
    print("Missing covariance", c, "spread risk", float(a @ K @ a))
print("This is an analytical identification illustration, not a market confidence interval.")

## Interpretation and next decision

Separate missing-pair ambiguity, basis restriction, conditioning, sampling and
numerical error. The market decomposition measures representation losses; it
does not observe the true SPDE covariance. Carry material residual risk forward
into a joint model rather than equating reconstructed coefficients with clean
market truth.

## Reproduction and research boundary

This notebook reads checksummed aggregate evidence, not raw exchange files.
The numerical algorithms are in `spde_covariance`; use the
[protocol](../docs/empirical-research-protocol.md) to rerun local calibration.
Execute this notebook with `uv run python scripts/check_notebook.py notebooks/02_representation_and_identification.ipynb`.

The original historical benchmark remains fixed. New analyses on the inspected
2024–2025 period are exploratory, even though forecast issuance is past-only.
No final untouched confirmation period, executable trading backtest or physical
forcing identification is claimed. See the [roadmap](../README.md).